<a href="https://colab.research.google.com/github/LuciaMellini/AMD_project/blob/main/findingSimilarItems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding similar items

We download the Letterboxd dataset from Kaggle, using a token.

In [1]:
import os
import json
import pandas as pd
import pip
import string
import re

os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"

/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:31: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


In [2]:
#! kaggle datasets download -d gsimonx37/letterboxd

We only consider a subset of the files contained in the `letterboxd` dataset, namely the data regarding the movie names and ids, their actors, crews, genres and themes.

In [3]:
# import zipfile
# from multiprocessing import Pool

DATA_DIR = "./letterboxd"
# members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
# with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
#     for file_name in members_to_extract:
#         zip_ref.extract(file_name + '.csv', DATA_DIR)


We then prepare the entry point for the Spark functionalities that will we use from now on.

In [4]:
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
#!wget https://downloads.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf spark-3.5.3-bin-hadoop3.tgz
!rm spark-3.5.3-bin-hadoop3.tgz
!pip install -q findspark

In [5]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = "spark-3.5.3-bin-hadoop3"

import findspark
findspark.init("spark-3.5.3-bin-hadoop3")
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

We begin by getting the input files into RDD form.

In [116]:
members_to_extract = ['actors', 'crew', 'genres', 'movies', 'themes']
letterboxd_RDDs={}

def extract_data(member):
    rdd = sc.textFile(DATA_DIR + "/" + member + ".csv")

    if member == 'actors':
        rdd = rdd.zipWithIndex().map(lambda r: r[0] + ',' + str(r[1]))
        rdd = rdd.map(lambda r: re.sub(r'(\d+),[\s\t]+([a-zA-Z])', r'\1,\2', r))
    rdd = rdd.map(lambda r: re.split(r',(?! )', r))  #split only on commas that are followed by a character to avoid splitting sentences e.g. in movie descriptions

    # get column name from csv different from 'id'
    column_names = rdd.filter(lambda r: r[0]=='id').collect()[0][1:]
    if member == 'actors':
        column_names[-1] = 'movie_n'
    rdd = (rdd
            .map(lambda r: (r[0], dict(zip(column_names, r[1:]))))
            .filter(lambda r: r[0]!='id'))
    return rdd

for member in members_to_extract:
    letterboxd_RDDs[member] = extract_data(member)


In [117]:
letterboxd_RDDs['themes'] = letterboxd_RDDs['themes'].mapValues(lambda x: {**x, 'theme': x['theme'].strip('"')})

Let's look at the amount of rows for each category.

In [71]:
for member in members_to_extract:
    print(f"Number of rows for {member}:\t{letterboxd_RDDs[member].count()}")

Number of rows for actors:	5798450
Number of rows for crew:	4720183
Number of rows for genres:	1046849
Number of rows for movies:	941597
Number of rows for themes:	125641


A glimpse at the structure of the rows in the RDDs of each category.

In [72]:
for member in members_to_extract:
    print(f"Row for {member:}:\t {letterboxd_RDDs[member].first()}")

Row for actors:	 ('1000001', {'name': 'Margot Robbie', 'role': 'Barbie', 'movie_n': '1'})
Row for crew:	 ('1000001', {'role': 'Director', 'name': 'Greta Gerwig'})
Row for genres:	 ('1000001', {'genre': 'Comedy'})
Row for movies:	 ('1000001', {'name': 'Barbie', 'date': '2023', 'tagline': "She's everything. He's just Ken.", 'description': '"Barbie and Ken are having the time of their lives in the colorful and seemingly perfect world of Barbie Land. However, when they get a chance to go to the real world, they soon discover the joys and perils of living among humans."', 'minute': '114', 'rating': '3.86'})
Row for themes:	 ('1000001', {'theme': 'Humanity and the world around us'})


Below we have prepared a function to extract a sample of the data, based on the ids in the datasets. The maximum size of the sample is $125641$.

In [118]:
def get_sample(rdd, size):
    return rdd.filter(lambda r: int(r[0],10)<=1000000+size)

for member in members_to_extract:
    letterboxd_RDDs[member] = get_sample(letterboxd_RDDs[member], 100)

In [74]:
print(letterboxd_RDDs['actors'].count())
print(letterboxd_RDDs['crew'].count())
print(letterboxd_RDDs['genres'].count())
print(letterboxd_RDDs['movies'].count())
print(letterboxd_RDDs['themes'].count())

6130
9115
275
100
704


For each category the available attributes are the following:

| **Category**    | **Attributes**                             |
|--------------|-----------------------------------------|
| **actor**    | name, role                          |
| **crew**     | role, name                          |
| **genres**   | genre                               |
| **movies**   | name, date, tagline, description, minute, rating |
| **themes**   | theme                               |

For this project we would like to focus on the following features:

| **Category**    | **Attributes**                             |
|--------------|-----------------------------------------|
| **actor**    | names of the first 10 actors for a given movie                      |
| **crew**     | name(s) of the director of each movie                        |
| **genres**   | genre                               |
| **movies**   | name, date, minute, rating |
| **themes**   | theme                               |


We keep only the 6 most relevant actors in each movie.

In [119]:
n_actors = 6
letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                            .groupByKey().map(lambda r: (r[0], list(r[1])))
                            .map(lambda r: (r[0], sorted(r[1], key=lambda x: x["movie_n"])[:n_actors]))
                            .map(lambda r: (r[0], list_dicts_to_dict(r[1]))))

In [120]:
def filter_dict_fields(d, final_fields):
    return {key: d[key] for key in final_fields if key in d}

def list_dicts_to_dict(l):
    return {key: [d[key] for d in l] for key in l[0]}

def remove_dict_field(d, field):
    del d[field]
    return d

def rename_key_in_dict(d, old_key, new_key):
    d[new_key] = d.pop(old_key)
    return d

def replace_dict_values(d, keys_to_replace, f):
    return {k: (f(v) if k in keys_to_replace else v) for k, v in d.items()}

For the actor we rename the relative field to "actors".

In [121]:
letterboxd_RDDs['actors'] = (letterboxd_RDDs['actors']
                            .map(lambda r: (r[0], filter_dict_fields(r[1], ['name'])))
                            .map(lambda r: (r[0], rename_key_in_dict(r[1], 'name', 'actors'))))

We filter only the directors from the crew dataset, and we ony keep their name.

In [122]:
letterboxd_RDDs['crew'] = (letterboxd_RDDs['crew']
                           .filter(lambda r: r[1]['role']=='Director')
                           .map(lambda r: (r[0], filter_dict_fields(r[1], ['name'])))
                           .map(lambda r: (r[0], rename_key_in_dict(r[1], 'name', 'director'))))

For each movie we only store the attributes listed in the table above.

In [123]:
letterboxd_RDDs['movies'] = (letterboxd_RDDs['movies']
                            .map(lambda r: (r[0], filter_dict_fields(r[1], ['name', 'date', 'minute', 'rating']))))

For each category we arrange for movies having multiple values for a given attribute.

In [124]:
for member in members_to_extract[1:]:
    letterboxd_RDDs[member] = (letterboxd_RDDs[member]
                            .groupByKey().map(lambda r: (r[0], list(r[1])))
                            .map(lambda r: (r[0], list_dicts_to_dict(r[1]))))


In [127]:
letterboxd_RDDs['crew'].take(5)

[('1000012', {'director': ['Damien Chazelle']}),
 ('1000017', {'director': ['Christopher Nolan']}),
 ('1000021',
  {'director': ['Joaquim Dos Santos', 'Justin K. Thompson', 'Kemp Powers']}),
 ('1000029', {'director': ['Michel Gondry']}),
 ('1000036', {'director': ['Quentin Tarantino']})]

In [41]:
movies_RDD = letterboxd_RDDs[members_to_extract[0]]
for member in members_to_extract[1:]:
    movies_RDD = movies_RDD.join(letterboxd_RDDs[member]).mapValues(lambda x: {**x[0], **x[1]})

In [42]:
movies_RDD.first()

('1000064',
 {'actors': ['Amy Adams',
   'Jeremy Renner',
   'Forest Whitaker',
   'Michael Stuhlbarg',
   'Tzi Ma',
   "Mark O'Brien"],
  'director': 'Denis Villeneuve',
  'genre': 'Science Fiction',
  'name': 'Arrival',
  'date': '2016',
  'minute': '116',
  'rating': '4.12',
  'theme': 'Monsters, aliens, sci-fi and the apocalypse'})

# Data pre-processing

To preserve the independent role of each attribute we have decided to measure their similarity using cosine distance. This entails translating each movie data dictionary into a vector in $\mathbb{R}^{17}$. In fact we will work in a space where each feature is a dimension, and we consider each of the 10 actors starring in the movie as a distinct attribute.



In [43]:
def list_to_dict(l, field_base_name):
    return {f"{field_base_name}{i + 1}": value for i, value in enumerate(l)}

movies_RDD = (movies_RDD.map(lambda r: (r[0], {**r[1],**list_to_dict(r[1]['actors'], 'actor')}))
                                        .map(lambda r: (r[0], remove_dict_field(r[1], 'actors'))))

In [44]:
letterboxd_RDDs['actors'].first()

('1000004',
 {'actors': ['Edward Norton',
   'Brad Pitt',
   'Helena Bonham Carter',
   'Meat Loaf',
   'Jared Leto',
   'Zach Grenier']})

Below we list the data types of the various attributes.

| **Attributes**    | **Datatype**                       |
|--------------|-----------------------------------------|
| **actors**    | string         |
| **director**     | string                        |
| **genre**   | string                             |
| **theme**   | string                             |
| **name**   | string |
| **date**   | numerical |
| **minute**   | numerical |
| **rating**   | numerical                             |

It is evident that the textual attributes have to be transformed into strings to be able to work in an Euclidean space. The following paragraphs are dedicated to these transformations. We refer to the report for a discussion regarding the chosen methods.


## String preprocessing

We bring all the strings in data dictionary to lower case, eccept for the names of actors and director. In addition to names always being capitalized, we will not consider them from a semantic point of view, so their uniformation in preparation for the next steps would be useless.

In [52]:
movies_RDD.count()

2125

In [45]:
categories_lower = ['genre', 'name', 'theme']
movies_lower_RDD = movies_RDD.map(lambda r: (r[0], replace_dict_values(r[1], categories_lower, lambda x: x.lower())))

To distill the semantics of the movie's theme we apply the following NLP processing steps:
* remove stop words
* replace the words with their lemmatized version

In [46]:
import spacy
! python -m spacy download en_core_web_md -q

nlp = spacy.load("en_core_web_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 12.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [50]:
from functools import reduce

combine_functions = lambda *funcs: lambda x: reduce(lambda v, f: f(v), funcs, x) #v accumulator value
remove_punctuation = lambda x: re.sub(r'[^\w\s]','',x)
remove_multiple_spaces = lambda x: re.sub(r'\s+',' ',x)
remove_stop_words_func = lambda x: " ".join([token.text for token in nlp(x) if not token.is_stop])
lemmatize_func = lambda x: " ".join([token.lemma_ for token in nlp(x)])

nlp_processing = combine_functions(remove_punctuation, remove_multiple_spaces, remove_stop_words_func, lemmatize_func)
print(nlp_processing("The quick brown fox jumps over the lazy dog"))

categories_nlp = ['theme']
print(movies_lower_RDD.count())
movies_nlp_RDD = movies_lower_RDD.map(lambda r: (r[0], replace_dict_values(r[1], categories_nlp, nlp_processing)))
print(movies_nlp_RDD.count())

quick brown fox jump lazy dog
2125
2125


In [51]:
movies_nlp_RDD.take(3)

[('1000031',
  {'director': 'Mark Mylod',
   'genre': 'comedy',
   'name': 'the menu',
   'date': '2022',
   'minute': '107',
   'rating': '3.54',
   'theme': 'intense violence sexual transgression',
   'actor1': 'Anya Taylor-Joy',
   'actor2': 'Ralph Fiennes',
   'actor3': 'Nicholas Hoult',
   'actor4': 'Janet McTeer',
   'actor5': 'Paul Adelstein',
   'actor6': 'Rob Yang'}),
 ('1000031',
  {'director': 'Mark Mylod',
   'genre': 'comedy',
   'name': 'the menu',
   'date': '2022',
   'minute': '107',
   'rating': '3.54',
   'theme': 'humanity world',
   'actor1': 'Anya Taylor-Joy',
   'actor2': 'Ralph Fiennes',
   'actor3': 'Nicholas Hoult',
   'actor4': 'Janet McTeer',
   'actor5': 'Paul Adelstein',
   'actor6': 'Rob Yang'}),
 ('1000031',
  {'director': 'Mark Mylod',
   'genre': 'comedy',
   'name': 'the menu',
   'date': '2022',
   'minute': '107',
   'rating': '3.54',
   'theme': 'twist dark psychological thriller',
   'actor1': 'Anya Taylor-Joy',
   'actor2': 'Ralph Fiennes',
   'a

### Embedding strings using word2vec

We apply word2vec embeddings to attributes:
* name
* genre
* theme

In [28]:
embedding_func = lambda x: nlp(x).vector

categories_word2vec = ['name', 'genre', 'theme']
movie_cat_embeddings_word2vec_RDD = movies_RDD.map(lambda r: (r[0], replace_dict_values(r[1], categories_word2vec, embedding_func)))

In [36]:
movie_cat_embeddings_word2vec_RDD.take(4)

[('1000031',
  {'director': 'Mark Mylod',
   'genre': array([-0.062557, -0.57405 , -2.2819  , -3.5485  , -2.865   , -0.47755 ,
           3.5966  , -1.1911  ,  1.877   , -6.3136  ,  0.11898 ,  0.48955 ,
           0.35902 , -0.067902,  0.56267 , -1.2421  ,  3.711   ,  1.7051  ,
           0.42196 ,  1.1366  ,  2.9756  ,  2.5019  ,  1.5689  , -0.73023 ,
           0.97205 , -0.74231 , -1.1203  ,  2.8286  ,  0.26411 , -1.2125  ,
          -5.645   ,  4.2858  ,  0.88063 ,  1.9466  , -3.3112  ,  5.2654  ,
          -2.558   , -1.1497  , -4.049   , -6.3927  , -0.1191  ,  0.11943 ,
           2.9965  , -0.82819 ,  1.6066  ,  0.31484 , -1.6343  , -1.1161  ,
           4.5843  ,  4.1737  , -4.076   ,  2.2131  , -0.092931,  1.2038  ,
          -0.693   ,  0.21445 , -1.9457  , -0.58546 , -0.85541 ,  5.5119  ,
          -3.7096  ,  1.5352  , -4.4012  , -0.14387 , -0.85866 , -0.15886 ,
          -6.7351  , -6.4101  , -3.363   , -1.7402  , -2.7163  ,  0.4983  ,
          -3.6139  , -5.0256  ,  1.13

### Hashing strings

We hash the strings of the features:
* actors
* director
person name hash
(no bias towards similar names)

In [30]:
import hashlib
hash_func = lambda x: int(hashlib.md5(x.encode()).hexdigest(), 16)

categories_hash = [f"actor{i+1}" for i in range(n_actors)]+['director']
movie_cat_embeddings_RDD = movie_cat_embeddings_word2vec_RDD.map(lambda r: (r[0], replace_dict_values(r[1], categories_hash, hash_func)))

## Vector preparation

Now that we have prepared all categories in a targeted way, we can proceed by building the vectors of the movies.

In [31]:
movie_cat_embeddings_RDD.first()

('1000002',
 {'director': 137525233821544095936720183130881750726,
  'genre': array([-0.062557, -0.57405 , -2.2819  , -3.5485  , -2.865   , -0.47755 ,
          3.5966  , -1.1911  ,  1.877   , -6.3136  ,  0.11898 ,  0.48955 ,
          0.35902 , -0.067902,  0.56267 , -1.2421  ,  3.711   ,  1.7051  ,
          0.42196 ,  1.1366  ,  2.9756  ,  2.5019  ,  1.5689  , -0.73023 ,
          0.97205 , -0.74231 , -1.1203  ,  2.8286  ,  0.26411 , -1.2125  ,
         -5.645   ,  4.2858  ,  0.88063 ,  1.9466  , -3.3112  ,  5.2654  ,
         -2.558   , -1.1497  , -4.049   , -6.3927  , -0.1191  ,  0.11943 ,
          2.9965  , -0.82819 ,  1.6066  ,  0.31484 , -1.6343  , -1.1161  ,
          4.5843  ,  4.1737  , -4.076   ,  2.2131  , -0.092931,  1.2038  ,
         -0.693   ,  0.21445 , -1.9457  , -0.58546 , -0.85541 ,  5.5119  ,
         -3.7096  ,  1.5352  , -4.4012  , -0.14387 , -0.85866 , -0.15886 ,
         -6.7351  , -6.4101  , -3.363   , -1.7402  , -2.7163  ,  0.4983  ,
         -3.6139  , -5.0

In [32]:
movie_vectors_RDD = movie_cat_embeddings_RDD.map(lambda r: (r[0], list(r[1].values())))

In [33]:
movie_vectors_RDD.first()

('1000098',
 [56975319645372228956191911258599963464,
  array([-3.6125  , -3.3777  ,  0.52097 , -2.1773  ,  0.73541 , -0.93237 ,
          3.7686  ,  1.5689  , -0.28429 , -2.184   ,  7.133   ,  4.6867  ,
         -6.1678  , -0.096264,  2.4102  ,  3.1364  ,  5.0721  ,  3.7309  ,
         -6.4435  , -0.90286 ,  1.1457  ,  3.087   , -2.0999  ,  1.8936  ,
          0.30309 , -2.0755  , -2.4272  , -1.5749  , -3.64    ,  1.643   ,
          0.06172 ,  2.1827  , -0.2436  , -0.23403 , -3.8059  , -1.6083  ,
          3.162   , -0.27413 ,  3.081   , -3.9065  ,  1.802   , -4.9001  ,
         -2.633   ,  1.144   , -0.82062 ,  1.4241  , -1.2761  , -3.5324  ,
         -0.85462 , -1.9361  , -1.8896  ,  5.5094  ,  0.96419 , -2.9847  ,
          0.10841 , -1.1893  , -1.6508  ,  2.9007  , -2.1375  , -4.1941  ,
         -0.076007,  0.032274, -3.7552  ,  2.2459  ,  2.7801  ,  3.1531  ,
         -3.5889  , -4.9465  ,  2.5193  ,  0.088085, -4.6679  ,  2.1067  ,
         -3.6314  , -2.5775  , -1.9913  , -0.2

## LSH hash functions

We build the locality sensitive family $\mathbf{F}$ as a set of randomly chosen vectors $\{v_{f\in\mathbf{F}}\}$. Given two vectors $x$ and $y$, they make a candidate pair of similar items if and only if the dot products $x\cdot v_f$ and $x \cdot v_f$ have the same sign. A family of functions $\mathbf{F}$ built as described is a locality-sensitive family for the cosine distance.

We will also refer to the random vectors in $\mathbf{F}$ as hash functions.

Since we will compute the hash functions of each of the elements in the dataset, given each of the hash functions in $\mathbf{F}$, we try to simplify the computation of the cosine distance between vectors. We do so by restricting the random choice of vectors to those having components $+1$ or $-1$. Hence the dot product of any vector $x$ with a vector in such a family $\mathbf{F}$ is given by its algebraic sum $x$'s components, where the signs depend on the components of the random vector.